# Generate Synthetic Samples with White Space Tokenization and Concept_ID free labels (for version v01)

# White Space Tokenization

In [ ]:
import json, re, csv
# Load the saved datasets back in
def load_jsonl(filename):
    with open(filename, "r") as f:
        return [json.loads(line) for line in f]
# Load synthetic data from version 0.0 
# These are just the text with symptom and polarity general labels

# THIS IS JUST TO SEE A COUPLE OF EXAMPLES
train_data = load_jsonl("../data_v00/synthetic_data.jsonl")
train_data[:2]


[{'text': 'There are no symptoms of stridor.',
  'symptom_id': 's0486',
  'is_negated': True},
 {'text': 'The patient has dry mouth.',
  'symptom_id': 's0712',
  'is_negated': False}]

In [9]:
# TOKEN_PATTERN explanation:
# - [A-Za-z0-9]+          : Match a sequence of one or more alphanumeric characters (a word)
# - (?:['-][A-Za-z0-9]+)* : Match zero or more groups where an apostrophe or hyphen is followed by more alphanumerics;
#                           This allows "can't", "mother-in-law", etc. to be tokenized as single words.
# - |                     : OR
# - [^\sA-Za-z0-9]        : Match any single character that is NOT whitespace and is NOT alphanumeric.
#                           This picks up standalone punctuation marks as their own tokens (e.g., ".", ",", "(", ")").
# The overall result: words, possibly including internal apostrophes/hyphens, are single tokens;
# all other non-alphanumeric non-whitespace characters are split as their own tokens.

TOKEN_PATTERN = re.compile(
    r"[A-Za-z0-9]+(?:['-][A-Za-z0-9]+)*|[^\sA-Za-z0-9]"
)

# ----------------------------
# Helpers
# ----------------------------
# Tokenizer that separates words and punctuation into separate tokens.
# Pattern: words with internal apostrophes or hyphens, or any single non-space punctuation char
TOKEN_PATTERN = re.compile(r"[A-Za-z0-9]+(?:['-][A-Za-z0-9]+)*|[^\sA-Za-z0-9]")

def tokenize_with_spans(text):
    """
    Return list of (token, start_char, end_char) using TOKEN_PATTERN.
    Example: "stridor." -> [("stridor", idx, idx+7), (".", idx+7, idx+8)]
    """
    tokens = []
    for m in TOKEN_PATTERN.finditer(text):
        tok = m.group(0)
        tokens.append((tok, m.start(), m.end()))
    return tokens

def normalize_token(tok):
    """Lowercase normalization for matching (leave punctuation tokens as-is)."""
    return tok.lower()

def find_subsequence(token_norms, target_tokens):
    """
    Find first index i where token_norms[i:i+len(target_tokens)] == target_tokens.
    Returns index or None.
    """
    n = len(target_tokens)
    if n == 0:
        return None
    for i in range(len(token_norms) - n + 1):
        ok = True
        for j in range(n):
            if token_norms[i + j] != target_tokens[j]:
                ok = False
                # Don't check the remaining tokens, if first one does not match, we cant match the remaining ones
                break
        if ok:
            return i
    return None

def symptom_to_tokenlist(symptom_text):
    """Convert symptom prefLabel to normalized token list (split on whitespace)."""
    # keep internal hyphens/apostrophes as part of tokens
    parts = [p for p in re.split(r"\s+", symptom_text.strip()) if p]
    parts_norm = [p.lower() for p in parts]
    return parts_norm

# ----------------------------
# Load symptom dictionary (id -> prefLabel)
# ----------------------------
symptom_map = {}
with open("../base_symptom_dict.csv", newline='', encoding='utf-8') as f:
    # assume CSV has header and column 'id' and 'prefLabel'
    reader = csv.DictReader(f)
    for r in reader:
        sid = r.get("id") or r.get("symptom_id") or r.get("ID")
        pref = r.get("prefLabel") or r.get("pref_label") or r.get("preflabel")
        if sid is None or pref is None:
            continue
        symptom_map[sid] = pref


In [13]:

# ----------------------------
# Process input JSONL
# ----------------------------
import os
os.makedirs("data", exist_ok=True)
out_f = open("data/synthetic_data_tokenized.jsonl", "w", encoding="utf-8")
not_found = []

with open("../data_v00/synthetic_data.jsonl", "r", encoding="utf-8") as fh:
    for line in fh:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        text = obj["text"]
        sid = obj["symptom_id"]
        is_neg = bool(obj.get("is_negated", False))

        # get symptom text from dict
        symptom_text = symptom_map.get(sid)
        if symptom_text is None:
            # unknown id - skip or record as O-only
            not_found.append({"line": line, "reason": "unknown_symptom_id"})
            print(f"WARNING: NO text was found for {sid}")
            continue

        # 1) Tokenize text into tokens with spans
        tokens_with_spans = tokenize_with_spans(text)
        tokens = [t for (t, s, e) in tokens_with_spans]
        token_norms = [normalize_token(t) for t in tokens]

        # 2) Build symptom token list (normalized)
        symptom_tokens = symptom_to_tokenlist(symptom_text)

        # 3) Try to find symptom as a subsequence in token_norms
        start_idx = find_subsequence(token_norms, symptom_tokens)

        # 4) Fallback: try matching by removing punctuation from token_norms ends (rare)
        if start_idx is None:
            # create versions with punctuation stripped from token ends
            stripped = [re.sub(r'^\W+|\W+$', '', t).lower() for t in tokens]
            start_idx = find_subsequence(stripped, symptom_tokens)

        # 6) If still not found, record and label entire example as O
        labels = ["O"] * len(tokens)
        if start_idx is not None:
            # generate labels B / I
            n = len(symptom_tokens)
            # Map POS/NEG suffix
            suffix = "NEG" if is_neg else "POS"
            b_label = f"B-SYMPTOM_{suffix}" # HERE IS THE DIFFERENCE! NO sid like in v00
            i_label = f"I-SYMPTOM_{suffix}"
            for k in range(n):
                pos = start_idx + k
                if pos < 0 or pos >= len(labels):
                    # safety: skip if out of range
                    continue
                labels[pos] = b_label if k == 0 else i_label
        else:
            not_found.append({"text": text, "symptom_id": sid, "symptom_text": symptom_text})

        # write out tokenized example
        out_obj = {
            "text": text,
            "word_tokens": tokens,
            "word_labels": labels,
            "symptom_id": sid,
            "is_negated": is_neg
        }
        out_f.write(json.dumps(out_obj, ensure_ascii=False) + "\n")

out_f.close()

# ----------------------------
# Report
# ----------------------------

print("Total not found / ambiguous matches:", len(not_found))
if len(not_found) > 0:
    # show a few
    print("Examples of not-found:")
    for e in not_found[:10]:
        print(e)

Total not found / ambiguous matches: 0
